In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("ModelRF_Deteksi_DDoS")
         .master("spark://192.168.56.9:7077")
         .getOrCreate()
        )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/10/07 06:02:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
csv_path = "hdfs://192.168.56.9:9000/user/hadoop/bigdata/dataset_sdn.csv"

df = (spark.read
      .option("header",True)
      .option("inferSchema",True)
      .csv(csv_path)
     )

In [3]:
df.describe().show()

+-------+------------------+------------------+--------+--------+-----------------+--------------------+------------------+--------------------+--------------------+-----------------+-----------------+-----------------+-----------------+------------------+-------------------+--------+------------------+-------------------+--------------------+-----------------+------------------+------------------+-------------------+
|summary|                dt|            switch|     src|     dst|         pktcount|           bytecount|               dur|            dur_nsec|             tot_dur|            flows|        packetins|       pktperflow|      byteperflow|           pktrate|           Pairflow|Protocol|           port_no|           tx_bytes|            rx_bytes|          tx_kbps|           rx_kbps|          tot_kbps|              label|
+-------+------------------+------------------+--------+--------+-----------------+--------------------+------------------+--------------------+------------

In [4]:
from pyspark.sql.functions import col, sum, count, lit

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

missing_df = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])

+---+------+---+---+--------+---------+---+--------+-------+-----+---------+----------+-----------+-------+--------+--------+-------+--------+--------+-------+-------+--------+-----+
| dt|switch|src|dst|pktcount|bytecount|dur|dur_nsec|tot_dur|flows|packetins|pktperflow|byteperflow|pktrate|Pairflow|Protocol|port_no|tx_bytes|rx_bytes|tx_kbps|rx_kbps|tot_kbps|label|
+---+------+---+---+--------+---------+---+--------+-------+-----+---------+----------+-----------+-------+--------+--------+-------+--------+--------+-------+-------+--------+-----+
|  0|     0|  0|  0|       0|        0|  0|       0|      0|    0|        0|         0|          0|      0|       0|       0|      0|       0|       0|      0|    506|     506|    0|
+---+------+---+---+--------+---------+---+--------+-------+-----+---------+----------+-----------+-------+--------+--------+-------+--------+--------+-------+-------+--------+-----+



In [5]:
from pyspark.sql.functions import col, sum, when

total_rows = df.count()
print(f"Jumlah rows adalah : {total_rows}")

Jumlah rows adalah : 104345


In [6]:
missing_counts = [count(when(col(c).isNull(),c)).alias(f'Jumlah_NULL_{c}')for c in df.columns]
missing_df = df.agg(*missing_counts)

In [7]:
if total_rows == 0:
    print("DataFrame kosong.")
else:
    print("Ringkasan Missing Values (NULL) per kolom:")

missing_row = missing_df.collect()[0]
print("{:<30} {:<15} {:<15}".format("Kolom", "Jumlah NULL", "Persentase (%)"))
print("="*60)
for c in df.columns:
    count_null = missing_row[f'Jumlah_NULL_{c}']
    percent_null = round((count_null / total_rows) * 100, 2)
    print("{:<30} {:<15}{:<15}".format(c, count_null, f"{percent_null}%"))

Ringkasan Missing Values (NULL) per kolom:


[Stage 11:===================>                                      (1 + 2) / 3]

Kolom                          Jumlah NULL     Persentase (%) 
dt                             0              0.0%           
switch                         0              0.0%           
src                            0              0.0%           
dst                            0              0.0%           
pktcount                       0              0.0%           
bytecount                      0              0.0%           
dur                            0              0.0%           
dur_nsec                       0              0.0%           
tot_dur                        0              0.0%           
flows                          0              0.0%           
packetins                      0              0.0%           
pktperflow                     0              0.0%           
byteperflow                    0              0.0%           
pktrate                        0              0.0%           
Pairflow                       0              0.0%           
Protoco

In [8]:
# Menghapus kolom missing values

kolom_missing = ["rx_kbps","tot_kbps"]
df_clean = df.na.drop(subset=kolom_missing)

In [9]:
print(f"Jumlah baris awal: {df.count()}")
print(f"Jumlah baris setelah di cleaning: {df_clean.count()}")

Jumlah baris awal: 104345


[Stage 17:======================================>                   (2 + 1) / 3]

Jumlah baris setelah di cleaning: 103839


In [10]:
# Menangani kolom kategorial
from pyspark.ml.feature import StringIndexer

Protocol_encoding = StringIndexer(inputCol="Protocol",outputCol="Protocol_encoding")
df_clean = Protocol_encoding.fit(df_clean).transform(df_clean)

print("\Pemetaan dari StringIndexer:")
df_clean.select("Protocol","Protocol_encoding").distinct().orderBy("Protocol_encoding").show()

\Pemetaan dari StringIndexer:


[Stage 23:===================>                                      (1 + 2) / 3]

+--------+-----------------+
|Protocol|Protocol_encoding|
+--------+-----------------+
|    ICMP|              0.0|
|     UDP|              1.0|
|     TCP|              2.0|
+--------+-----------------+



In [15]:
from pyspark.ml.feature import VectorAssembler

kolom_fitur = ['dt','dur','dur_nsec','tot_dur','pktrate','Protocol_encoding','port_no','tx_kbps','rx_kbps','tot_kbps']
TARGET = 'label'
df_clean = df_clean.withColumn(TARGET, col(TARGET).cast("double"))
                               
assembler = VectorAssembler (inputCols=kolom_fitur,outputCol="features")
df_vector = assembler.transform(df_clean)
df_final = df_vector.withColumnRenamed(TARGET, 'label')


In [16]:
train_data, test_data = df_final.randomSplit([0.8, 0.2],seed=42)

print(f"Jumlah data train: {train_data.count()}")
print(f"Jumlah data test: {test_data.count()}")

Jumlah data train: 83185


[Stage 35:======================================>                   (2 + 1) / 3]

Jumlah data test: 20654


In [17]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol='label',featuresCol='features', numTrees=100, seed=42)
model = rf.fit(train_data)

In [19]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

prediksi = model.transform(test_data)
print("Hasil prediksi pada data testing (5 baris teratas):")
prediksi.select('label', 'prediction', 'probability').show(5, truncate=False)

Hasil prediksi pada data testing (5 baris teratas):


[Stage 55:>                                                         (0 + 1) / 1]

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |0.0       |[0.9045224771672439,0.0954775228327561] |
|0.0  |0.0       |[0.9068843574738268,0.09311564252617317]|
|0.0  |0.0       |[0.9045224771672439,0.0954775228327561] |
|0.0  |0.0       |[0.9045224771672439,0.0954775228327561] |
|0.0  |0.0       |[0.9045224771672439,0.0954775228327561] |
+-----+----------+----------------------------------------+
only showing top 5 rows



In [20]:
prediksi.select('label','prediction','probability')

evaluator = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
akurasi = evaluator.evaluate(prediksi)
print(f"Akurasi model:{akurasi}")

[Stage 56:======================================>                   (2 + 1) / 3]

Akurasi model:0.9120751428294761


In [24]:
from pyspark.mllib.evaluation import MulticlassMetrics
predictionAndLabels = prediksi.select(['prediction','label']).rdd.map(tuple)

metrics = MulticlassMetrics(predictionAndLabels)
cm = metrics.confusionMatrix().toArray()
print("\nConfusion Matrix:")
print(cm)

/home/ubaid/.local/lib/python3.10/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
[Stage 60:======================================>                   (2 + 1) / 3]


Confusion Matrix:
[[11211.  1415.]
 [  401.  7627.]]


In [29]:
import os
MODEL_PATH = os.path.expanduser("~/random_forest_model_local")
model.write().overwrite().save(MODEL_PATH)


In [23]:
!pwd


/home/ubaid
